In [1]:
import time
import numpy as np
from MultiLevel_NUADM import MultiLevel_NUADM

# Teste MultiLevel NUADM

## Funções auxiliares

In [2]:
def generate_adj_cent(H, V, A):
    nb=[H, V, A]
    lb=[1, 1, 1]
    sp=[0.0, 0.0, 0.0]
    ms = np.mgrid[sp[0]+0.5*lb[0]:sp[0]+(nb[0]+0.5)*lb[0]:lb[0],
                  sp[1]+0.5*lb[1]:sp[1]+(nb[1]+0.5)*lb[1]:lb[1],
                  sp[2]+0.5*lb[2]:sp[2]+(nb[2]+0.5)*lb[2]:lb[2]]

    ms = ms.flatten()
    centroids = ms.reshape(3,int(ms.size/3)).T
    ############
    # ijk0 = np.array([centroids[:, 0]//lb[0], centroids[:, 1]//lb[1], centroids[:, 2]//lb[2]])
    # ee = ijk0[0] + ijk0[1]*nb[0] + ijk0[2]*nb[0]*nb[1] #60 3600
    # # import pdb; pdb.set_trace()
    # centroids=centroids[ee.astype(int)]
    ############
    GID_0 = np.arange(len(centroids))
    adjs0 = np.tile(GID_0,(np.array(nb)>1).sum())
    adjs1 = np.array([])
    sz=1
    sy=1
    if nb[2]>1:
        adjs1 = np.concatenate([adjs1, GID_0+1])
        sz=nb[2]
    if nb[1]>1:
        adjs1 = np.concatenate([adjs1, GID_0+sz])
        sy=nb[1]
    if nb[0]>1:
        adjs1 = np.concatenate([adjs1, GID_0+sz*sy])
    adjs1=adjs1.astype(int)
    adjs=np.vstack([adjs0,adjs1]).T
    adjs=adjs[adjs.max(axis=1)<=GID_0.max()]
    dif=abs(centroids[adjs[:,0]]-centroids[adjs[:,1]])
    adjacencies=adjs[((dif>0).sum(axis=1)==1) & ((dif<=lb).sum(axis=1)==3)]
    areas=np.array([lb[1]*lb[2], lb[0]*lb[2], lb[0]*lb[1]])
    areas=np.tile(areas,len(adjacencies)).reshape(len(adjacencies),3)[abs(centroids[adjacencies[:,0]]-centroids[adjacencies[:,1]])>0]
    return centroids, adjacencies

In [3]:
def label_levels_from_fines(adjs, fines, max_level):
    gids = np.arange(adjs.max() + 1)
    levels = np.full_like(gids, max_level + 1)  # valores default maiores que o nível máximo
    levels[fines] = 0  # nível inicial (fines)

    current_level = 0
    current_front = set(fines)

    while current_level < max_level:
        # Encontra adjacências onde pelo menos um vértice está no nível atual
        mask = np.isin(adjs[:, 0], list(current_front)) | np.isin(adjs[:, 1], list(current_front))
        neighbors = adjs[mask].flatten()

        # Remove os que já foram rotulados com nível menor ou igual
        next_front = np.setdiff1d(neighbors, np.where(levels <= current_level)[0], assume_unique=False)

        # Atribui novo nível
        levels[next_front] = current_level + 1

        # Prepara próxima iteração
        current_front = set(next_front)
        current_level += 1

        if len(current_front) == 0:
            break  # Não há mais vizinhos para expandir

    return levels

## Processamento


In [4]:
H = 10
V = 10
A = 35
nX = [3, 3, 3, 3]
nY = [3, 3, 3, 3]
nZ = [5, 3, 3, 3]

fines = np.random.choice(H*V*A, size=round(H*V*A * 0.3), replace=False) # 30% dos volumes são finos
centroids, adjs = generate_adj_cent(H, V, A)
gids = np.arange(H*V*A)
vector = label_levels_from_fines(adjs, gids[fines],1) # 5 níveis

In [5]:
nuadm = MultiLevel_NUADM(nX, nY, nZ, levels=3)
init = time.time()
nuadm.run(centroids, vector)
end = time.time()
print(f'Tempo de processamento: {end - init:.4f} segundos')
nuadm.validate()
duals = nuadm.get_duals()
print(f'Duals generated: {len(duals)}')
for i in range(len(duals)):
    print(f'Level {i} dual shape: {duals[i].shape}')

done generating levels
ids initialized
Tempo de processamento: 0.0111 segundos
Cells in level 0: 3500 - 10.0x10.0x35.0
Cells in level 1: 112 - 4x4x7
Cells in level 2: 12 - 2x2x3
NU-ADM IDs length: 3500
Duals generated: 2
Level 0 dual shape: (3500,)
Level 1 dual shape: (3500,)


## Plot para visualizar no VisIt

In [9]:
import numpy as np
import gstools as gs
import pyvista as pv
import scipy.sparse as sp

def get_grid():
    ordem=[2,1,0]
    nb=np.array([nx, ny, nz])[ordem]+1
    lb=np.array([hx,hy,hz])[ordem]
    sp=np.array([0,0,0])
    grid = pv.ImageData()
    grid.dimensions = nb
    grid.origin = sp  # The bottom left corner of the data set
    grid.spacing = lb  # These are the cell sizes along each axis
    return grid
# Dimensões SPE10
# nx, ny, nz = 60, 220, 85
# Lx, Ly, Lz = 1200.0, 2200.0, 850.0
nx, ny, nz = H, V, A
Lx, Ly, Lz = H, V, A
hx, hy, hz = Lx/nx, Ly/ny, Lz/nz

x = np.linspace(hx/2, Lx - hx/2, nx)
y = np.linspace(hy/2, Ly - hy/2, ny)
z = np.linspace(hz/2, Lz - hz/2, nz)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Modelo geoestatístico
model = gs.Exponential(
    dim=3,
    var=1.0,
    len_scale=[300, 300, 3],
    anis=[1.0, 1.0, 0.05],
)

srf = gs.SRF(model, seed=42)
log_k = srf((X, Y, Z))
log_k = 6 * (log_k - np.mean(log_k)) / np.std(log_k)
k = np.exp(log_k)  # mD

# Salvar como .txt
k_flat = k.flatten(order='F')
grid=get_grid()
centroids=np.array([X.flatten(),Y.flatten(),Z.flatten()]).T

# grid.cell_data["ks"] = np.log10(k_flat)
grid.cell_data['color'] = np.ones_like(vector)
grid.cell_data['levels'] = vector
grid.cell_data['NUADM_ids'] = nuadm.id_NUADM
grid.cell_data['FINA'] = nuadm.fine_mesh
for i in range(len(duals)):
    grid.cell_data[f'DUAL{i}'] = duals[i]
    grid.cell_data[f'PRIMAL{i}'] = nuadm.mapping[i]


# grid.cell_data["ks"] = np.log10(k_flat)



grid.save('arquivos/results/example_mesh.vtk') #ativar
# import pdb; pdb.set_trace()
# np.savetxt("permeabilidade_spe10_geoestatistica.txt", k_flat, fmt="%.6e")
# import pdb; pdb.set_trace()


## Plot com matplotlib

In [7]:
# import matplotlib.pyplot as plt
# from matplotlib.patches import Patch
# from matplotlib.colors import ListedColormap, BoundaryNorm
# from skimage.measure import label, find_contours
# from matplotlib.patches import Rectangle

# def draw_grid(ax, x0, y0, H, V, edge_color='black'):
#     """Desenha apenas as bordas dos quadrados internos (sem preencher)."""
#     cell_size = 1
#     for i in range(H):
#         for j in range(V):
#             ax.add_patch(Rectangle(
#                 (x0 + i, y0 + j),
#                 cell_size, cell_size,
#                 facecolor='none',  # Deixa transparente
#                 edgecolor=edge_color,
#                 linewidth=0.5
#             ))
# comp_mesh = 1
# lvls = levels.reshape((H, V))
# fina = sla.fine_mesh.reshape((H, V))
# comp = sla.mapping[comp_mesh].reshape((H, V))
# id_nuadm = sla.id_NUADM.reshape((H, V))
# dual = sla.get_duals()[comp_mesh].reshape((H, V))
# zeros = np.zeros_like(lvls)
# ones = np.ones_like(lvls)
# twos = np.full_like(lvls, 2)

# fig, ax = plt.subplots(figsize=(7, 7))

# # Definindo valores únicos e cores a partir de colormap
# colors = ["#0000ff", "#00ffff", "#00ff00", "#ff0000"]
# labels = [f'Nível {v}' for v in range(3)]

# # Criando o colormap com as cores definidas
# cores = ListedColormap(colors)
# norm = BoundaryNorm(boundaries=np.arange(-0.5, len(cores.colors), 1), ncolors=len(cores.colors))

# # Mostrar a malha com as cores
# cax = ax.imshow(dual, cmap=cores, norm=norm, origin='upper', extent=[0, V, H, 0])
# # cax = ax.imshow(comp, cmap="inferno", origin='upper', extent=[0, V, H, 0])

# # # Desenhar as bordas dos quadrados
# # draw_grid(ax, 0, 0, H, V)

# # Criar patches da legenda
# legend_elements = [Patch(facecolor=cor, label=label, edgecolor='black') 
#                    for cor, label in zip(colors, labels)]

# # Colocar legenda fora do gráfico, na lateral direita
# ax.legend(handles=legend_elements, title='Níveis',
#           loc='center left', bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)


# # # Adicionar os números em cada célula
# # for i in range(H):
# #     for j in range(V):
# #         if lvls[i, j] == comp_mesh + 1:
# #             ax.text(j + 0.5, i + 0.5, str(sla.id_NUADM[i * H + j]),
# #                     ha='center', va='center', color='black', fontsize=8)

# # Para cada valor único da malha, encontra e desenha as bordas
# for i in range(H):
#     for j in range(V):
#         val = dual[i, j]
#         # Cima
#         if i > 0 and dual[i - 1, j] != val:
#             ax.plot([j, j + 1], [i, i], color='black', linewidth=1)
#         elif i == 0:
#             ax.plot([j, j + 1], [i, i], color='black', linewidth=1)
#         # Baixo
#         if i == H - 1 or dual[i + 1, j] != val:
#             ax.plot([j, j + 1], [i + 1, i + 1], color='black', linewidth=1)
#         # Esquerda
#         if j > 0 and dual[i, j - 1] != val:
#             ax.plot([j, j], [i, i + 1], color='black', linewidth=1)
#         elif j == 0:
#             ax.plot([j, j], [i, i + 1], color='black', linewidth=1)
#         # Direita
#         if j == V - 1 or dual[i, j + 1] != val:
#             ax.plot([j + 1, j + 1], [i, i + 1], color='black', linewidth=1)


# # Remover os eixos
# ax.set_xlim(0, V)
# ax.set_ylim(H, 0)
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_xticklabels([])
# ax.set_yticklabels([])
# # ax.set_xlabel(V, fontsize=14)
# # ax.set_ylabel(H, fontsize=14, rotation=0, labelpad=10)   
# ax.tick_params(left=False, bottom=False)

# # Exibir grade
# ax.grid(False)

# plt.title(f'Malha NU-ADM', fontsize=16)
# plt.tight_layout()
# plt.show()

In [8]:
# import matplotlib.pyplot as plt
# from matplotlib.patches import Rectangle

# def draw_grid(ax, x0, y0, total_size, n_cells, edge_color='gray'):
#     """Desenha apenas as bordas dos quadrados internos (sem preencher)."""
#     cell_size = total_size / n_cells
#     for i in range(n_cells):
#         for j in range(n_cells):
#             ax.add_patch(Rectangle(
#                 (x0 + i * cell_size, y0 + j * cell_size),
#                 cell_size, cell_size,
#                 facecolor='none',  # Deixa transparente
#                 edgecolor=edge_color,
#                 linewidth=0.5
#             ))


# # Criar figura e eixo
# fig, ax = plt.subplots(figsize=(6, 6))

# # Coordenadas x de onde colocar os quadrados
# x_positions = [0, 3, 10]
# sizes = [1, 5, 15]  # tamanho relativo de cada quadrado (1x1, 3x3, 9x9 unidades)
# labels = ['Nível 0', 'Nível 1\n(5x5)', 'Nível 2\n(15x15)']
# colors = ["#0000ff", "#00ffff", "#00ff00"]

# # Níveis internos (grade de quantos quadrados por lado)
# subdivs = [1, 5, 15]

# # Desenhar quadrados com subdivisões internas
# for x, size, color, label, n in zip(x_positions, sizes, colors, labels, subdivs):
#     # Quadrado externo
#     ax.add_patch(Rectangle((x, 0), size, size, facecolor=color, edgecolor='black', linewidth=1.5))
#     # Subdivisões internas
#     if n > 1:
#         draw_grid(ax, x, 0, size, n)
#     # # Texto
#     # ax.text(x + size / 2, -0.5, label, ha='center', va='top', fontsize=13)

# # Ajustes de visual
# ax.set_xlim(-1, x_positions[-1] + sizes[-1] + 1)
# ax.set_ylim(-1, sizes[-1] + 1)
# ax.set_aspect('equal')
# ax.axis('off')
# # plt.title('Razões de Engrossamento', fontsize=16)
# plt.tight_layout()
# plt.show()
